In [ ]:
!pip install onnxruntime
!pip install onnx --upgrade
!pip install biotite
!pip install scipy
!pip install einops
!pip install h5py
!pip install -e .

In [ ]:
!pip install 'dllogger @ git+https://github.com/NVIDIA/dllogger.git'
!pip install 'openfold @ git+https://github.com/aqlaboratory/openfold.git@4b41059694619831a7db195b7e0988fc4ff3a307'

In [2]:
import torch
import esm
model = esm.pretrained.esmfold_structure_module_only_150M()
model = model.eval()

/home/siria/anaconda3/envs/esmfold2/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/siria/anaconda3/envs/esmfold2/lib/python3.9/site-packages/openfold/model/primitives.py:33: UserWarning: A NumPy version >=1.22.4 and <2.3.0 is required for this version of SciPy (detected version 1.21.2)
  from scipy.stats import truncnorm
/home/siria/esm/esm/pretrained.py:215: UserWarning: Regression weights not found, predicting contacts will not produce correct results.
  warnings.warn(


In [ ]:
sequence = "MKTVRQERLKSIVRILERSKEPVSGAQLAEELSVSRQVIVQDIAYLRSLGYNIVATPRGYVLAGG"
# Multimer prediction can be done with chains separated by ':'
model.cuda()
with torch.no_grad():
    output = model.infer_pdb(sequence)

with open("result.pdb", "w") as f:
    f.write(output)

import biotite.structure.io as bsio
struct = bsio.load_structure("result.pdb", extra_fields=["b_factor"])
print(struct.b_factor.mean())  # this will be the pLDDT
# 88.3

1
{'frames': torch.Size([8, 1, 68, 7]), 'sidechain_frames': torch.Size([8, 1, 68, 8, 4, 4]), 'unnormalized_angles': torch.Size([8, 1, 68, 7, 2]), 'angles': torch.Size([8, 1, 68, 7, 2]), 'positions': torch.Size([8, 1, 68, 14, 3]), 'states': torch.Size([8, 1, 68, 384]), 's_s': torch.Size([1, 68, 1024]), 's_z': torch.Size([1, 68, 68, 128])}
72.25544502617801


In [7]:
import torch
import torch.nn as nn
from typing import Dict


class ESMLanguageModelWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.esmfold = model

    def _mask_inputs_to_esm(self, esmaa, mask):
        """
        Replace `pattern == 1` with ONNX-supported operations.
        """
        new_esmaa = esmaa.clone()
        new_esmaa[mask==1] = self.esmfold.esm_dict.mask_idx  # Supported by ONNX
        return new_esmaa

    def forward(
        self,
        aa: torch.Tensor
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass for the ESM language model and preprocessing.
        """
        
        mask = torch.ones_like(aa)

        # === ESM Language Model ===
        esmaa = self.esmfold._af2_idx_to_esm_idx(aa, mask)

        esm_s, esm_z = self.esmfold._compute_language_model_representations(esmaa)
        esm_s = esm_s.to(self.esmfold.esm_s_combine.dtype)
        esm_s = (self.esmfold.esm_s_combine.softmax(0).unsqueeze(0) @ esm_s).squeeze(2)

        # === Preprocessing ===
        s_s_0 = self.esmfold.esm_s_mlp(esm_s)
        if self.esmfold.cfg.use_esm_attn_map:
            esm_z = esm_z.to(self.esmfold.esm_s_combine.dtype)
            s_z_0 = self.esmfold.esm_z_mlp(esm_z)
        else:
            s_z_0 = s_s_0.new_zeros(aa.shape[0], aa.shape[1], aa.shape[1],
                                   self.esmfold.cfg.trunk.pairwise_state_dim)

        s_s_0 += self.esmfold.embedding(aa)

        return {
            "s_s_0": s_s_0,
            "s_z_0": s_z_0,
        }

In [8]:
import torch
import esm

# Create dummy inputs
batch_size, seq_len = 1, 1024
aa = torch.randint(0, 20, (batch_size, seq_len), dtype=torch.long)

# Initialize the wrapper
esm_lm_wrapper = ESMLanguageModelWrapper(model)
esm_lm_wrapper.eval().to("cpu")
#esm_lm_wrapper(aa)
# Export to ONNX
torch.onnx.export(
    esm_lm_wrapper,
    aa,
    "esm_lm.onnx",
    export_params=True,
    do_constant_folding=True,
    input_names=["aa"],
    output_names=["s_s_0", "s_z_0"],
    dynamic_axes={
        "aa": {0:"batch",1: "seq_len"}
    },
    opset_version=17
)

/home/siria/esm/esm/model/esm2.py:108: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if not padding_mask.any():
/home/siria/esm/esm/multihead_attention.py:193: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert embed_dim == self.embed_dim
/home/siria/esm/esm/multihead_attention.py:194: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  as

: 

In [3]:
import onnxruntime as ort
import numpy as np

# Load the ONNX model
onnx_session = ort.InferenceSession("esm_lm.onnx")
esm_lm_wrapper = ESMLanguageModelWrapper(model)
esm_lm_wrapper.eval().to("cuda").half()

batch_size, seq_len = 1, 1024
aa = torch.randint(0, 20, (batch_size, seq_len), dtype=torch.long)
# Prepare the input for the ONNX model
onnx_input = {onnx_session.get_inputs()[0].name: aa.to("cpu").numpy()}

# Run inference with the ONNX model
onnx_output = onnx_session.run(None, onnx_input)
# Run inference with the PyTorch model
with torch.no_grad():
    pytorch_output = esm_lm_wrapper(aa.to("cuda"))
pytorch_output = [pytorch_output["s_s_0"].to("cpu").numpy(),pytorch_output["s_z_0"].to("cpu").numpy()]

# Calculate the absolute difference between the outputs
diff = np.abs(pytorch_output[0] - onnx_output[0])

# Print statistics about the difference
print(f"Max difference: {np.max(diff)}")
print(f"Mean difference: {np.mean(diff)}")
print(f"Number of differences > 1e-5: {np.sum(diff > 1e-4)}")

diff = np.abs(pytorch_output[1] - onnx_output[1])

# Print statistics about the difference
print(f"Max difference: {np.max(diff)}")
print(f"Mean difference: {np.mean(diff)}")
print(f"Number of differences > 1e-5: {np.sum(diff > 1e-4)}")

Max difference: 0.5991349220275879
Mean difference: 0.04148979112505913
Number of differences > 1e-5: 1046718
Max difference: 1.2709770202636719
Mean difference: 0.004631367977708578
Number of differences > 1e-5: 130816560


In [3]:
import torch
import torch.nn as nn

class Distogram(nn.Module):
    def __init__(self, min_bin, max_bin, num_bins):
        super().__init__()
        self.min_bin = min_bin
        self.max_bin = max_bin
        self.num_bins = num_bins

    def forward(self, coords):
        boundaries = torch.linspace(self.min_bin, self.max_bin, self.num_bins - 1, device=coords.device)
        boundaries = boundaries**2
        N, CA, C = [x.squeeze(-2) for x in coords.chunk(3, dim=-2)]
        b = CA - N
        c = C - CA
        a = b.cross(c, dim=-1)
        CB = -0.58273431 * a + 0.56802827 * b - 0.54067466 * c + CA
        dists = (CB[..., None, :, :] - CB[..., :, None, :]).pow(2).sum(dim=-1, keepdims=True)
        bins = torch.sum(dists > boundaries, dim=-1)
        return bins

# Export Distogram
distogram = Distogram(min_bin=3.375, max_bin=21.375, num_bins=15)
coords = torch.randn(1, 100, 3, 3)  # Example input

class trunkWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        
        self.recycle_s_norm = model.trunk.recycle_s_norm 
        self.recycle_z_norm = model.trunk.recycle_z_norm 
        self.recycle_disto = model.trunk.recycle_disto 
        self.pairwise_positional_embedding=model.trunk.pairwise_positional_embedding
        self.structure_module = model.trunk.structure_module
        self.trunk2sm_s = model.trunk.trunk2sm_s
        self.trunk2sm_z = model.trunk.trunk2sm_z
        self.distogram = Distogram(min_bin=3.375, max_bin=21.375, num_bins=15)

    
    def forward(self,true_aa, s_s_0, s_z_0, recycle_s, recycle_z, recycle_bins, residx, mask):
        def trunk_iter(s, z, residx, mask):
            z = z + self.pairwise_positional_embedding(residx, mask=mask)
            return s, z

        recycle_s = self.recycle_s_norm(recycle_s.detach())
        recycle_z = self.recycle_z_norm(recycle_z.detach())
        recycle_z += self.recycle_disto(recycle_bins.detach())
        s_s, s_z = trunk_iter(s_s_0 + recycle_s, s_z_0 + recycle_z, residx, mask)
        structure = self.structure_module(
            {"single": self.trunk2sm_s(s_s), "pair":  self.trunk2sm_z(s_z)},
            true_aa
        )
        recycle_s = s_s
        recycle_z = s_z
        recycle_bins=self.distogram(structure["positions"][-1][:, :, :3])
        return structure, s_s, s_z, recycle_s, recycle_z, recycle_bins

# Export Recycling
true_aa = torch.randint(0, 20, (1, 100))
s = torch.randn(1, 100, 1024)
z = torch.randn(1, 100, 100, 128)
trunk = trunkWrapper(model).to("cpu")
recycle_s = torch.randn(1, 100, 1024)  # Example input
recycle_z = torch.randn(1, 100, 100, 128) # Example input
recycle_bins = torch.randint(0, 15, (1, 100, 100))  # Example input
residx = torch.arange(100).unsqueeze(0)  # Shape: [batch_size, sequence_length]
mask = torch.ones(1, 100)  # Shape: [batch_size, sequence_length]


torch.onnx.export(
    trunk,
    (true_aa, s, z, recycle_s, recycle_z, recycle_bins,residx, mask),
    "structure_module_new.onnx",
    input_names=["aa","s_s_0","s_z_0","recycle_s", "recycle_z", "recycle_bins","residx","mask"],
    output_names=['frames', 'sidechain_frames', 'unnormalized_angles', 'angles', 'positions', 'states', 'single',"s_s","s_z", "updated_recycle_s", "updated_recycle_z","updated_recycle_bins"],
    dynamic_axes={
        "aa": {0:"batch",1: "sequence_length"},
        "s_s_0": {0:"batch",1: "sequence_length"},
        "s_z_0": {0:"batch",1: "sequence_length", 2: "sequence_length"},
        "recycle_s": {0:"batch",1: "sequence_length"},
        "recycle_z": {0:"batch",1: "sequence_length", 2: "sequence_length"},
        "recycle_bins": {0:"batch",1: "sequence_length", 2: "sequence_length"},
        "residx": {0:"batch",1: "sequence_length"},
        "mask": {0:"batch",1: "sequence_length"},
    },
)

/home/siria/esm/esm/esmfold/v1/trunk.py:96: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  assert residue_index.shape == mask.shape
/home/siria/anaconda3/envs/esmfold2/lib/python3.9/site-packages/openfold/utils/rigid_utils.py:312: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if((rot_mats is not None and rot_mats.shape[-2:] != (3, 3)) or
/home/siria/anaconda3/envs/esmfold2/lib/python3.9/site-packages/openfold/utils/rigid_utils.py:854: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data f

In [4]:
import torch
import torch.nn as nn
from typing import Dict
import torch
import torch.nn as nn
from typing import Dict
from openfold.data.data_transforms import make_atom14_masks
from esm.esmfold.v1.categorical_mixture import categorical_lddt

from openfold.utils.loss import compute_predicted_aligned_error, compute_tm

import torch
import torch.nn as nn
from typing import Dict

class PostProcessingWrapper(nn.Module):
    def __init__(self, esmfold_model):
        super().__init__()
        self.esmfold = esmfold_model
        self.distogram_bins = 64
        self.lddt_bins = 50

    def forward(
        self,
        aa,
        structure
    ) -> Dict[str, torch.Tensor]:
        """
        Forward pass for post-processing.
        """
        B, L=aa.shape
        mask = torch.ones_like(aa)
        residx = torch.arange(aa.shape[1], device=aa.device).expand_as(aa)

        # === Distogram Head ===
        disto_logits = self.esmfold.distogram_head(structure["s_z"])
        disto_logits = (disto_logits + disto_logits.transpose(1, 2)) / 2
        structure["distogram_logits"] = disto_logits

        lm_logits = self.esmfold.lm_head(structure["s_s"])
        structure["lm_logits"] = lm_logits

        structure["aatype"] = aa
        make_atom14_masks(structure)
        for k in [
            "atom14_atom_exists",
            "atom37_atom_exists",
        ]:
            structure[k] *= mask.unsqueeze(-1)
        structure["residue_index"] = residx

        lddt_head = self.esmfold.lddt_head(structure["states"]).reshape(
            structure["states"].shape[0], B, L, -1, self.lddt_bins
        )
        structure["lddt_head"] = lddt_head
        plddt = categorical_lddt(lddt_head[-1], bins=self.lddt_bins)
        structure["plddt"] = (
            100 * plddt
        )  # we predict plDDT between 0 and 1, scale to be between 0 and 100.

        ptm_logits = self.esmfold.ptm_head(structure["s_z"])

        seqlen = mask.type(torch.int64).sum(1)
        structure["ptm_logits"] = ptm_logits
        structure["ptm"] = torch.stack(
            [
                compute_tm(
                    batch_ptm_logits[None, :sl, :sl],
                    max_bins=31,
                    no_bins=self.distogram_bins,
                )
                for batch_ptm_logits, sl in zip(ptm_logits, seqlen)
            ]
        )
        structure.update(
            compute_predicted_aligned_error(
                ptm_logits, max_bin=31, no_bins=self.distogram_bins
            )
        )

        return structure

In [5]:
import torch
import esm

# Create an instance of the wrapper
post_processing_wrapper = PostProcessingWrapper(model)

# Set the model to evaluation mode
post_processing_wrapper.eval().to("cpu")

# Create dummy inputs for tracing
B, L = 1, 100  # Batch size and sequence length
aa = torch.randint(0, 20, (B, L))  # Amino acid indices
structure={'frames': torch.randn([8, B, L, 7]),
           'sidechain_frames': torch.randn([8, B, L, 8, 4, 4]),
           'unnormalized_angles': torch.randn([8, B, L, 7, 2]), 
           'angles': torch.randn([8, B, L, 7, 2]), 
           'positions': torch.randn([8, B, L, 14, 3]), 
           'states': torch.randn([8, B, L, 384]), 
           's_s': torch.randn([B, L,model.cfg.trunk.sequence_state_dim]), 
           's_z': torch.randn([B, L, L,model.cfg.trunk.pairwise_state_dim])}
# Export to ONNX
onnx_file_path = "post_processing.onnx"
torch.onnx.export(
    post_processing_wrapper,
    (aa,structure,{}),
    onnx_file_path,
    export_params=True,
    opset_version=17,
    do_constant_folding=True,
    input_names=[
        "aa",
        'frames_0',
        'sidechain_frames_0',
        'unnormalized_angles_0', 
        'angles_0', 
        'positions_0', 
        'states_0', 
        's_s_0', 
        's_z_0'],
    output_names=[
        'frames', 'sidechain_frames', 'unnormalized_angles', 'angles',
        'positions', 'states', 's_s', 's_z', 'distogram_logits', 'lm_logits', 
        'aatype', 'atom14_atom_exists', 'residx_atom14_to_atom37', 
        'residx_atom37_to_atom14', 'atom37_atom_exists', 'residue_index', 
        'lddt_head', 'plddt', 'ptm_logits', 'ptm', 'aligned_confidence_probs', 
        'predicted_aligned_error', 'max_predicted_aligned_error'
        ],
    dynamic_axes={
        "aa": {0: "batch_size", 1: "seq_len"},
        'frames_0': {1: "batch_size", 2: "seq_len"},
        'sidechain_frames_0': {1: "batch_size", 2: "seq_len"},
        'unnormalized_angles_0': {1: "batch_size", 2: "seq_len"}, 
        'angles_0': {1: "batch_size", 2: "seq_len"}, 
        'positions_0': {1: "batch_size", 2: "seq_len"}, 
        "states_0": {1: "batch_size", 2: "seq_len"},
        "s_s_0": {0: "batch_size", 1: "seq_len"},
        "s_z_0": {0: "batch_size", 1: "seq_len", 2: "seq_len"},
    },
)

print(f"Model exported to {onnx_file_path}")

/home/siria/anaconda3/envs/esmfold2/lib/python3.9/site-packages/openfold/data/data_transforms.py:616: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  restype_atom14_to_atom37 = torch.tensor(
/home/siria/anaconda3/envs/esmfold2/lib/python3.9/site-packages/openfold/data/data_transforms.py:621: TracerWarning: torch.tensor results are registered as constants in the trace. You can safely ignore this warning if you use this function to create tensors out of constant variables that would be the same every time you call this function. In any other case, this might cause the trace to be incorrect.
  restype_atom37_to_atom14 = torch.tensor(
/home/siria/anaconda3/envs/esmfold2/lib/python3.9/site-packages/openfold/data/data_transforms.py:626: Tr

Model exported to post_processing.onnx


In [5]:
# Assuming `esmfold_model` is already defined and has the required attributes
post_processing_wrapper = PostProcessingWrapper(model) # Move to GPU if necessary

# Create example inputs
B, L = 1, 100  # Batch size and sequence length
aa = torch.randint(0, 20, (B, L))  # Amino acid indices
structure={'frames': torch.randn([8, B, L, 7]),
           'sidechain_frames': torch.randn([8, B, L, 8, 4, 4]),
           'unnormalized_angles': torch.randn([8, B, L, 7, 2]), 
           'angles': torch.randn([8, B, L, 7, 2]), 
           'positions': torch.randn([8, B, L, 14, 3]), 
           'states': torch.randn([8, B, L, 384]), 
           's_s': torch.randn([B, L,model.cfg.trunk.sequence_state_dim]), 
           's_z': torch.randn([B, L, L, model.cfg.trunk.pairwise_state_dim])}

# Run inference with the PyTorch model
with torch.no_grad():
    pytorch_output = post_processing_wrapper(aa,structure)
pytorch_output.keys()

dict_keys(['frames', 'sidechain_frames', 'unnormalized_angles', 'angles', 'positions', 'states', 's_s', 's_z', 'distogram_logits', 'lm_logits', 'aatype', 'atom14_atom_exists', 'residx_atom14_to_atom37', 'residx_atom37_to_atom14', 'atom37_atom_exists', 'residue_index', 'lddt_head', 'plddt', 'ptm_logits', 'ptm', 'aligned_confidence_probs', 'predicted_aligned_error', 'max_predicted_aligned_error'])